# DSA 504 — Class 17
## Shell Scripting II: Text Processing

**Date:** Monday, Nov 2
**Reading:** Course notes

---

> **If a `%%bash` cell fails** with something like `bash: command not found`, this usually means Git Bash isn't on your system's PATH yet (common on Windows). Fix: find where Git Bash is installed (commonly `C:\Program Files\Git\bin`), add that folder to your System PATH environment variable, then **fully restart** Jupyter (not just the kernel) before trying again. If it still doesn't work, come to office hours rather than losing class time to it.

Class 15 covered navigating, creating, and redirecting with the shell. Today covers the tools built specifically for **searching and transforming text** — some of the most powerful, most commonly used command-line tools in real data work.

### Learning goals
By the end of this class, you will be able to:
- Search text for a pattern with `grep`, including case-insensitive and counted searches
- Extract specific columns from delimited text with `cut`
- Find and count unique values with `sort` + `uniq`
- Perform a simple find-and-replace with `sed`
- Print, filter, and summarize fields with `awk`
- Chain several of these tools together into one pipeline


In [23]:
%%bash
cd ~
mkdir -p dsa504_class17
cd dsa504_class17
cat > sales_log.txt << 'EOF'
2026-01-05,Utica,electronics,4521.75
2026-01-05,Albany,groceries,980.10
2026-01-06,Rome,clothing,1102.00
2026-01-06,Utica,toys,340.00
2026-01-07,Syracuse,electronics,2210.50
2026-01-07,Albany,electronics,1875.25
2026-01-08,Utica,groceries,615.40
2026-01-08,Rome,toys,290.00
2026-01-09,Syracuse,clothing,875.60
2026-01-09,Albany,groceries,1020.00
2026-01-10,Utica,electronics,3390.00
2026-01-10,Rome,groceries,740.15
EOF
cat sales_log.txt
wc -l sales_log.txt

2026-01-05,Utica,electronics,4521.75
2026-01-05,Albany,groceries,980.10
2026-01-06,Rome,clothing,1102.00
2026-01-06,Utica,toys,340.00
2026-01-07,Syracuse,electronics,2210.50
2026-01-07,Albany,electronics,1875.25
2026-01-08,Utica,groceries,615.40
2026-01-08,Rome,toys,290.00
2026-01-09,Syracuse,clothing,875.60
2026-01-09,Albany,groceries,1020.00
2026-01-10,Utica,electronics,3390.00
2026-01-10,Rome,groceries,740.15
12 sales_log.txt


Every cell below `cd`s into `~/dsa504_class17` explicitly at the start — do the same in your own terminal, since a working directory doesn't necessarily carry over between separate command sessions.


## 1. `grep` — Searching Text for a Pattern

`grep` ("global regular expression print") searches a file line by line and prints every line that matches a pattern. If "pattern matching" sounds familiar, it should — this is the exact same idea as the regex work from Class 7, just applied to files and the terminal instead of pandas columns.


In [2]:
%%bash
cd ~/dsa504_class17
grep "electronics" sales_log.txt

2026-01-05,Utica,electronics,4521.75
2026-01-07,Syracuse,electronics,2210.50
2026-01-07,Albany,electronics,1875.25
2026-01-10,Utica,electronics,3390.00


In [3]:
%%bash
cd ~/dsa504_class17
grep -c "electronics" sales_log.txt

4


**`-c` counts matches instead of printing them.**


In [4]:
%%bash
cd ~/dsa504_class17
grep -i "UTICA" sales_log.txt

2026-01-05,Utica,electronics,4521.75
2026-01-06,Utica,toys,340.00
2026-01-08,Utica,groceries,615.40
2026-01-10,Utica,electronics,3390.00


**`-i` makes the search case-insensitive** — without it, `grep "UTICA"` would find nothing, since the file has "Utica," not "UTICA."


## 2. `cut` — Extracting Columns

`cut` pulls out specific fields from delimited text — the shell's equivalent of selecting one or two columns from a pandas DataFrame.


In [5]:
%%bash
cd ~/dsa504_class17
cut -d',' -f2 sales_log.txt

Utica
Albany
Rome
Utica
Syracuse
Albany
Utica
Rome
Syracuse
Albany
Utica
Rome


In [6]:
%%bash
cd ~/dsa504_class17
cut -d',' -f2,4 sales_log.txt

Utica,4521.75
Albany,980.10
Rome,1102.00
Utica,340.00
Syracuse,2210.50
Albany,1875.25
Utica,615.40
Rome,290.00
Syracuse,875.60
Albany,1020.00
Utica,3390.00
Rome,740.15


**`-d` sets the delimiter** (here, a comma) and **`-f` chooses which field(s)** — fields are numbered starting at 1, not 0.


## 3. `sort` and `uniq` — Finding Unique Values and Counting Them

You saw basic `sort` in Class 15. `uniq` removes adjacent duplicate lines — which is why it's almost always used *after* `sort`, since `uniq` only catches duplicates that are next to each other.


In [7]:
%%bash
cd ~/dsa504_class17
cut -d',' -f2 sales_log.txt | sort | uniq

Albany
Rome
Syracuse
Utica


In [8]:
%%bash
cd ~/dsa504_class17
cut -d',' -f2 sales_log.txt | sort | uniq -c

      3 Albany
      3 Rome
      2 Syracuse
      4 Utica


**`uniq -c`** counts how many times each unique value appears.


In [9]:
%%bash
cd ~/dsa504_class17
sort -t',' -k4 -n -r sales_log.txt

2026-01-05,Utica,electronics,4521.75
2026-01-10,Utica,electronics,3390.00
2026-01-07,Syracuse,electronics,2210.50
2026-01-07,Albany,electronics,1875.25
2026-01-06,Rome,clothing,1102.00
2026-01-09,Albany,groceries,1020.00
2026-01-05,Albany,groceries,980.10
2026-01-09,Syracuse,clothing,875.60
2026-01-10,Rome,groceries,740.15
2026-01-08,Utica,groceries,615.40
2026-01-06,Utica,toys,340.00
2026-01-08,Rome,toys,290.00


**`-t','`** sets the field separator, **`-k4`** sorts by the 4th field, **`-n`** tells `sort` to compare numerically (not alphabetically — without this, "900" would sort before "80", since "9" comes before "8" as *text*), and **`-r`** reverses the order to descending.


## 4. `sed` — Find and Replace

`sed` ("stream editor") transforms text as it passes through — most commonly used for find-and-replace.


In [10]:
%%bash
cd ~/dsa504_class17
sed 's/electronics/Electronics/' sales_log.txt

2026-01-05,Utica,Electronics,4521.75
2026-01-05,Albany,groceries,980.10
2026-01-06,Rome,clothing,1102.00
2026-01-06,Utica,toys,340.00
2026-01-07,Syracuse,Electronics,2210.50
2026-01-07,Albany,Electronics,1875.25
2026-01-08,Utica,groceries,615.40
2026-01-08,Rome,toys,290.00
2026-01-09,Syracuse,clothing,875.60
2026-01-09,Albany,groceries,1020.00
2026-01-10,Utica,Electronics,3390.00
2026-01-10,Rome,groceries,740.15


The pattern `s/old/new/` means "**s**ubstitute `old` with `new`." This prints the *modified* version to your screen — it does **not** change the actual file unless you add the `-i` flag (for "in place").

**A word of caution about `-i`:** `sed -i 's/electronics/Electronics/' sales_log.txt` would permanently overwrite the file with the replacement — there's no undo, same as the `rm` warning from Class 15. Always test a `sed` command *without* `-i` first, confirm the output looks right, and only add `-i` once you're sure.


## 5. `awk` — Field-Based Processing

`awk` is the most powerful of these tools — it treats each line as a set of fields and lets you print, filter, and calculate with them, almost like a tiny built-in programming language.


In [11]:
%%bash
cd ~/dsa504_class17
awk -F',' '{print $2, $4}' sales_log.txt

Utica 4521.75
Albany 980.10
Rome 1102.00
Utica 340.00
Syracuse 2210.50
Albany 1875.25
Utica 615.40
Rome 290.00
Syracuse 875.60
Albany 1020.00
Utica 3390.00
Rome 740.15


**`-F','`** sets the field separator. Inside the quotes, **`$2`** and **`$4`** refer to the 2nd and 4th fields — note `awk` fields are also 1-indexed, and `$0` always refers to the *whole* line.


In [12]:
%%bash
cd ~/dsa504_class17
awk -F',' '$4 > 2000 {print $2, $4}' sales_log.txt

Utica 4521.75
Syracuse 2210.50
Utica 3390.00


This is the shell equivalent of a pandas boolean filter (`sales[sales["revenue"] > 2000]`) or a SQL `WHERE` clause — same underlying idea, third syntax.


In [13]:
%%bash
cd ~/dsa504_class17
awk -F',' '{sum += $4} END {print "Total:", sum}' sales_log.txt

Total: 17960.8


Here, `{sum += $4}` runs once per line, adding that line's revenue to a running total; `END {print ...}` runs once, after every line has been processed, and prints the final result. (The total prints as `17960.8`, not `17960.75` — that's just `awk`'s default number formatting, not a calculation error.)


## 6. Combining Everything into One Pipeline

Real shell work often chains several of these tools together — each one doing one small job, piped into the next.

**Question: what are the top 3 highest-revenue electronics transactions, showing just store and revenue?**


In [14]:
%%bash
cd ~/dsa504_class17
grep "electronics" sales_log.txt | sort -t',' -k4 -n -r | cut -d',' -f2,4 | head -n 3

Utica,4521.75
Utica,3390.00
Syracuse,2210.50


**Reading this left to right:** find electronics rows → sort them by revenue, highest first → keep only the store and revenue columns → take just the top 3. Each tool does one job; the pipe (`|`) connects them into a single pipeline, with no intermediate files needed.


---
## Guided Practice

Work through these in the cells below, or in your own Terminal/Git Bash/WSL.


### Exercise 1 — grep
Find all rows for the `groceries` category. How many are there?


In [15]:
%%bash
# Exercise 1 — your commands here
cd ~/dsa504_class17



### Exercise 2 — cut + sort + uniq
List the unique categories in the file (hint: category is field 3).


In [16]:
%%bash
# Exercise 2 — your commands here
cd ~/dsa504_class17



### Exercise 3 — awk
Print only the rows where `store` is `Utica` (hint: `awk` can filter on any field, including text fields — use `$2 == "Utica"`).


In [17]:
%%bash
# Exercise 3 — your commands here
cd ~/dsa504_class17



### Exercise 4 (stretch) — Combine everything
Find the single highest-revenue row for the `groceries` category, showing only the date and revenue.


In [18]:
%%bash
# Exercise 4 — your commands here
cd ~/dsa504_class17



---
## Solutions

Try each exercise yourself first.


In [19]:
%%bash
# Solution — Exercise 1
cd ~/dsa504_class17
grep "groceries" sales_log.txt
grep -c "groceries" sales_log.txt

2026-01-05,Albany,groceries,980.10
2026-01-08,Utica,groceries,615.40
2026-01-09,Albany,groceries,1020.00
2026-01-10,Rome,groceries,740.15
4


In [20]:
%%bash
# Solution — Exercise 2
cd ~/dsa504_class17
cut -d',' -f3 sales_log.txt | sort | uniq

clothing
electronics
groceries
toys


In [21]:
%%bash
# Solution — Exercise 3
cd ~/dsa504_class17
awk -F',' '$2 == "Utica" {print}' sales_log.txt

2026-01-05,Utica,electronics,4521.75
2026-01-06,Utica,toys,340.00
2026-01-08,Utica,groceries,615.40
2026-01-10,Utica,electronics,3390.00


In [22]:
%%bash
# Solution — Exercise 4
cd ~/dsa504_class17
grep "groceries" sales_log.txt | sort -t',' -k4 -n -r | cut -d',' -f1,4 | head -n 1

2026-01-09,1020.00


---
## Wrap-up

**Recap:** `grep` for pattern matching (the same regex thinking from Class 7, now applied to files); `cut` for extracting columns; `sort`/`uniq` for finding and counting unique values; `sed` for find-and-replace (with a real warning about `-i`); `awk` for printing, filtering, and summarizing fields; and chaining all of these into a single pipeline.

**Common mistakes to watch for:**
- Forgetting `-n` when sorting numbers, causing `sort` to compare them as text (so "9" sorts before "80")
- Using `sed -i` before confirming the replacement looks right without it — there's no undo
- Off-by-one errors with field numbers — both `cut` and `awk` count fields starting at 1, not 0
- Forgetting the delimiter flag (`-d` for `cut`, `-F` for `awk`) on data that isn't space-separated

**Before next class (Nov 4):** Class 18 covers shell automation — turning a sequence of commands like today's into a reusable script — plus a brief comparison to PowerShell for Windows users. **HW5 is assigned that day.**
